# Required Libraries and Functions

In [0]:
import glob, re, os, requests

In [0]:
dbutils.widgets.text("volume_path", "/Volumes/workspace/taxi_weather/datasets/")

base_path = dbutils.widgets.get("volume_path")

# Get latest published data

In [0]:
# 1. Scan the Databricks Volume to get current file states
trip_files = glob.glob(f"{base_path}yellow_tripdata_*.parquet")


# 2. Extract YYYY-MM boundaries from the physical file names
months = [re.search(r'(\d{4}-\d{2})', f).group(1) for f in trip_files]
max_month = max(months)
min_month = min(months)

# 3. Calculate the next expected month
year, month = map(int, max_month.split('-'))
next_year = year + 1 if month == 12 else year
next_month = 1 if month == 12 else month + 1

target_month_str = f"{next_year}-{next_month:02d}"
target_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{target_month_str}.parquet"
target_file_path = f"{base_path}yellow_tripdata_{target_month_str}.parquet"

print(f"Current newest data: {max_month}. Checking TLC for: {target_month_str}...")

# 4. Ping the source endpoint
response = requests.get(target_url, stream=True)

if response.status_code == 200:
    print(f"New data found! Downloading {target_month_str}...")
    
    # 5. Download directly to volume in chunks
    with open(target_file_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    print("Download complete.")
    
    # 6. Drop the oldest month to maintain the rolling window
    oldest_file_path = f"{base_path}yellow_tripdata_{min_month}.parquet"
    if os.path.exists(oldest_file_path):
        os.remove(oldest_file_path)
        print(f"Dropped oldest data: {min_month}")
        

else:
    # 7. Halt pipeline execution if target data is not yet published
    dbutils.jobs.taskValues.set(key="new_data_found", value="false")
    print(f"Data for {target_month_str} not yet available. Status Code: {response.status_code}")
    dbutils.notebook.exit("No new data available. Halting pipeline.")

In [0]:
dbutils.jobs.taskValues.set(key="new_data_found", value="true")